In [3]:
from __future__ import division
import numpy as np
import sys

X_train = np.genfromtxt('X_train.csv', delimiter=",")
y_train = np.genfromtxt('y_train.csv')
X_test = np.genfromtxt('X_test_all.csv', delimiter=",")



In [ ]:
## can make more functions if required


def pluginClassifier(X_train, y_train, X_test):    
  # this function returns the required output 
    class_count = 10

    n = np.unique(y_train, return_counts=True)[1]
    pi = n / y_train.shape[0]
    mask = np.array([[y == class_index for y in y_train] for class_index in range(class_count)]) * 1

    mu = np.zeros((class_count, X_train.shape[1]))
    for class_index in range(class_count):
        mu[class_index] = np.sum(mask[class_index].reshape(-1,1) * X_train, axis=0) / n[class_index]
    
    sigma = np.zeros((class_count, X_train.shape[1], X_train.shape[1]))

    for class_index in range(class_count):
        for i in range(mask[class_index].shape[0]):
            if mask[class_index][i] > 0:
                sigma[class_index] += np.matmul(
                    (mu[class_index] - X_train[i]).reshape(-1, 1), 
                    (mu[class_index] - X_train[i]).reshape(1, -1))

        sigma[class_index] = sigma[class_index] / n[class_index]
    
    sigma_inv = np.array([np.linalg.inv(sigma[i]) for i in range(sigma.shape[0])])
    sigma_inv_sqrt_det = np.sqrt(
        np.array(
            [np.linalg.det(np.linalg.inv(sigma[i])) for i in range(sigma.shape[0])])
    )
    
    prob = np.zeros((X_test.shape[0], class_count))
    
    for i in range(X_test.shape[0]):
        for class_index in range(class_count):
            prob[i][class_index] = (pi[class_index] / sigma_inv_sqrt_det[class_index]) * \
                np.exp(-0.5 * np.linalg.multi_dot([
                (X_test[i] - mu[class_index]).reshape(1, -1),
                sigma_inv[class_index],
                (X_test[i] - mu[class_index]).reshape(-1, 1)
            ])[0][0])

    return prob / np.sum(prob, axis = 1).reshape(-1, 1)


final_outputs = pluginClassifier(X_train, y_train, X_test) # assuming final_outputs is returned from function

